In [4]:
import os
import sys
import json
import logging
from datetime import datetime
from typing import Optional, Dict, Any
import requests

# 配置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class DailyReportGenerator:
    """日报生成器主类"""
    
    def __init__(self, api_type: str = "openai", api_key: Optional[str] = None, 
                 base_url: Optional[str] = None, model: Optional[str] = None):
        """
        初始化生成器
        :param api_type: openai/ollama/qwen/doubao
        :param api_key: API密钥
        :param base_url: 自定义API地址
        :param model: 模型名称
        """
        self.api_type = api_type.lower()
        self.api_key = api_key or os.environ.get(f"{api_type.upper()}_API_KEY")
        
        # 设置默认配置
        self.config = self._get_default_config(base_url, model)
        
        # 验证配置
        if self.api_type != "ollama" and not self.api_key:
            print(f"⚠️ 警告: {api_type} API密钥未设置，将使用模拟模式")
            self.api_key = "dummy"
    
    def _get_default_config(self, base_url: Optional[str], model: Optional[str]) -> Dict:
        """获取默认配置"""
        configs = {
            "openai": {
                "base_url": base_url or "https://api.openai.com/v1",
                "model": model or "gpt-3.5-turbo",
                "headers": {"Authorization": f"Bearer {self.api_key}"}
            },
            "ollama": {
                "base_url": base_url or "http://localhost:11434",
                "model": model or "qwen2.5:7b",
                "headers": {}
            },
            "qwen": {
                "base_url": base_url or "https://dashscope.aliyuncs.com/compatible-mode/v1",
                "model": model or "qwen-turbo",
                "headers": {"Authorization": f"Bearer {self.api_key}"}
            },
            "doubao": {
                "base_url": base_url or "https://ark.cn-beijing.volces.com/api/v3",
                "model": model or "doubao-lite-32k",
                "headers": {"Authorization": f"Bearer {self.api_key}"}
            }
        }
        return configs.get(self.api_type, configs["openai"])
    
    def _build_prompt(self, raw_text: str, report_type: str = "daily") -> str:
        """
        构建提示词
        :param raw_text: 原始工作记录
        :param report_type: daily/weekly
        """
        sections = {
            "daily": ["今日完成", "进行中工作", "明日计划", "风险与备注"],
            "weekly": ["本周完成", "进行中工作", "下周计划", "风险与备注", "需要支持"]
        }
        
        section_titles = sections.get(report_type, sections["daily"])
        section_template = "\n".join([f"## {title}\n- " for title in section_titles])
        
        prompt = f"""你是一个专业的工作日报助手。请将以下零散的工作记录整理成规范的{report_type}报告。

要求：
1. 严格基于原始记录，不要捏造不存在的工作内容
2. 每条工作要点用"- "开头
3. 语言专业、简洁、得体
4. 如果没有某项内容，写"无"
5. 必须按以下格式输出，不要添加额外说明

输出格式：
{section_template}

原始记录：
{raw_text}

请直接输出整理后的报告，不要有其他内容。"""
        
        return prompt
    
    def _call_api_with_retry(self, prompt: str, max_retries: int = 3) -> Optional[str]:
        """
        调用API并支持重试
        """
        for attempt in range(max_retries):
            try:
                if self.api_type == "ollama":
                    return self._call_ollama(prompt)
                else:
                    return self._call_openai_compatible(prompt)
            except Exception as e:
                print(f"⚠️ API调用失败 (尝试 {attempt + 1}/{max_retries}): {str(e)}")
                if attempt == max_retries - 1:
                    print("🔄 使用本地兜底规则...")
                    return self._fallback_process(prompt)
        return None
    
    def _call_openai_compatible(self, prompt: str) -> str:
        """调用OpenAI兼容API"""
        url = f"{self.config['base_url']}/chat/completions"
        headers = self.config['headers'].copy()
        headers["Content-Type"] = "application/json"
        
        payload = {
            "model": self.config["model"],
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.3,
            "max_tokens": 1000
        }
        
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        response.raise_for_status()
        
        result = response.json()
        return result["choices"][0]["message"]["content"]
    
    def _call_ollama(self, prompt: str) -> str:
        """调用本地Ollama"""
        url = f"{self.config['base_url']}/api/generate"
        payload = {
            "model": self.config["model"],
            "prompt": prompt,
            "stream": False
        }
        
        response = requests.post(url, json=payload, timeout=60)
        response.raise_for_status()
        
        result = response.json()
        return result.get("response", "")
    
    def _fallback_process(self, prompt: str) -> str:
        """
        本地兜底规则（当API不可用时）
        """
        print("📋 使用本地关键词规则处理...")
        
        # 提取原始文本
        raw_text = prompt.split("原始记录：")[-1].split("\n请直接输出")[0].strip()
        
        # 关键词分类规则
        categories = {
    "今日完成": ["修复", "完成", "开发", "实现", "学习了", "写了", "做了", "完成", "上线", "部署", "提交", "解决"],
    "进行中工作": ["进行", "正在", "继续", "优化", "重构", "调试", "处理中"],
    "明日计划": ["计划", "明天", "准备", "安排", "规划", "即将", "打算"],
    "风险与备注": ["风险", "问题", "困难", "阻塞", "延迟", "注意", "阻塞", "延期"]
        }
        
        report = []
        for category, keywords in categories.items():
            items = []
            # 按句子分割
            for line in raw_text.split("\n"):
                for part in line.split("，"):
                    if any(kw in part for kw in keywords):
                        items.append(f"- {part.strip()}")
            if not items:
                items = ["- 无"]
            report.append(f"## {category}\n" + "\n".join(items))
        
        return "\n\n".join(report)
    
    def generate(self, raw_text: str, report_type: str = "daily") -> Dict[str, Any]:
        """
        生成报告
        :param raw_text: 原始工作记录
        :param report_type: daily/weekly
        :return: 包含报告和元数据的字典
        """
        print(f"🚀 开始生成{report_type}报告...")
        
        if not raw_text or len(raw_text.strip()) < 5:
            return {
                "success": False,
                "error": "输入内容太短，请提供更详细的工作记录",
                "report": None
            }
        
        prompt = self._build_prompt(raw_text, report_type)
        result = self._call_api_with_retry(prompt)
        
        if result:
            return {
                "success": True,
                "report": result.strip(),
                "metadata": {
                    "api_type": self.api_type,
                    "model": self.config["model"],
                    "timestamp": datetime.now().isoformat(),
                    "input_length": len(raw_text),
                    "raw_input": raw_text[:200] + ("..." if len(raw_text) > 200 else "")
                }
            }
        else:
            return {
                "success": False,
                "error": "所有API调用都失败，请检查网络或API配置",
                "report": None
            }

print("✅ DailyReportGenerator 类定义成功！")

✅ DailyReportGenerator 类定义成功！


In [3]:
# 创建生成器实例
generator = DailyReportGenerator(api_type="openai")

print("📌 生成器信息:")
print(f"   - API 类型: {generator.api_type}")
print(f"   - 模型: {generator.config['model']}")
print(f"   - API URL: {generator.config['base_url']}")

# 测试基本功能
test_input = "修复了登录页面的bug，下午和产品经理开了需求评审会，学习了Docker容器化部署"

print("\n📝 原始输入:")
print(test_input)
print("\n" + "="*60)

result = generator.generate(test_input)

if result["success"]:
    print("\n📊 生成的日报:")
    print(result["report"])
    print("\n" + "="*60)
    print(f"📌 元数据: {result['metadata']}")
else:
    print(f"❌ 错误: {result['error']}")

⚠️ 警告: openai API密钥未设置，将使用模拟模式
📌 生成器信息:
   - API 类型: openai
   - 模型: gpt-3.5-turbo
   - API URL: https://api.openai.com/v1

📝 原始输入:
修复了登录页面的bug，下午和产品经理开了需求评审会，学习了Docker容器化部署

🚀 开始生成daily报告...
⚠️ API调用失败 (尝试 1/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
⚠️ API调用失败 (尝试 2/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
⚠️ API调用失败 (尝试 3/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
🔄 使用本地兜底规则...
📋 使用本地关键词规则处理...

📊 生成的日报:
## 今日完成
- 修复了登录页面的bug
- 学习了Docker容器化部署

## 进行中工作
- 无

## 明日计划
- 无

## 风险与备注
- 无

📌 元数据: {'api_type': 'openai', 'model': 'gpt-3.5-turbo', 'timestamp': '2026-06-04T12:15:41.224750', 'input_length': 41, 'raw_input': '修复了登录页面的bug，下午和产品经理开了需求评审会，学习了Docker容器化部署'}


In [5]:
# 测试不同类型的工作记录
test_cases = [
    "修复了3个bug，写了单元测试，明天准备发布",
    "正在开发支付接口，遇到了技术难题，需要后端支持",
    "完成需求评审，计划下周开始编码，风险是工期紧张",
    "学习了FastAPI框架，部署了测试环境，帮同事review代码"
]

for i, test in enumerate(test_cases, 1):
    print(f"\n【测试 {i}】")
    print("-" * 40)
    result = generator.generate(test)
    if result["success"]:
        print(result["report"])
    print("="*40)


【测试 1】
----------------------------------------
🚀 开始生成daily报告...
⚠️ API调用失败 (尝试 1/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
⚠️ API调用失败 (尝试 2/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
⚠️ API调用失败 (尝试 3/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
🔄 使用本地兜底规则...
📋 使用本地关键词规则处理...
## 今日完成
- 修复了3个bug
- 写了单元测试

## 进行中工作
- 无

## 明日计划
- 明天准备发布

## 风险与备注
- 无

【测试 2】
----------------------------------------
🚀 开始生成daily报告...
⚠️ API调用失败 (尝试 1/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
⚠️ API调用失败 (尝试 2/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
⚠️ API调用失败 (尝试 3/3): 401 Client Error: Unauthorized for url: https://api.openai.com/v1/chat/completions
🔄 使用本地兜底规则...
📋 使用本地关键词规则处理...
## 今日完成
- 正在开发支付接口

## 进行中工作
- 正在开发支付接口

## 明日计划
- 无

## 风险与备注
- 无

【测试 3】
---------------------------------

In [1]:
streamlit_app_code = '''
import streamlit as st
import re
from datetime import datetime

# 页面配置
st.set_page_config(
    page_title="AI 日报生成器",
    page_icon="🤖",
    layout="wide"
)

# 自定义CSS
st.markdown("""
<style>
    .main-header {
        background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
        padding: 2rem;
        border-radius: 10px;
        margin-bottom: 2rem;
        text-align: center;
        color: white;
    }
    .report-box {
        background-color: #f8f9fa;
        padding: 1.5rem;
        border-radius: 10px;
        border-left: 4px solid #667eea;
        margin: 1rem 0;
        font-family: monospace;
        white-space: pre-wrap;
    }
    @media (max-width: 768px) {
        .main-header h1 { font-size: 1.5rem; }
    }
</style>
""", unsafe_allow_html=True)

class DailyReportGenerator:
    def __init__(self):
        self.keyword_map = {
            "今日完成": ["修复", "完成", "开发", "实现", "写了", "做了", "上线", "部署", 
                        "提交", "解决", "合并", "发布", "搞定", "学习", "阅读"],
            "进行中工作": ["正在", "进行", "继续", "优化", "重构", "调试", "处理"],
            "明日计划": ["明天", "计划", "准备", "安排", "规划", "即将", "打算"],
            "风险与备注": ["风险", "问题", "困难", "阻塞", "延迟", "注意"]
        }
    
    def generate(self, raw_text: str, report_type: str = "daily"):
        if not raw_text or len(raw_text.strip()) < 5:
            return {"success": False, "error": "输入内容太短"}
        
        sentences = re.split('[，,。；;！!？?、\\n]', raw_text)
        categorized = {cat: [] for cat in self.keyword_map.keys()}
        
        for sentence in sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
            assigned = False
            for category, keywords in self.keyword_map.items():
                if any(kw in sentence for kw in keywords):
                    categorized[category].append(f"- {sentence}")
                    assigned = True
                    break
            if not assigned:
                categorized["今日完成"].append(f"- {sentence}")
        
        output = []
        sections = ["今日完成", "进行中工作", "明日计划", "风险与备注"]
        for section in sections:
            if categorized[section]:
                unique = list(dict.fromkeys(categorized[section]))
                output.append(f"## {section}\\n" + "\\n".join(unique))
            else:
                output.append(f"## {section}\\n- 无")
        
        return {
            "success": True,
            "report": "\\n\\n".join(output),
            "metadata": {"timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
        }

generator = DailyReportGenerator()

# 界面
st.markdown('<div class="main-header"><h1>🤖 AI 智能日报生成器</h1><p>让工作汇报更简单</p></div>', unsafe_allow_html=True)

# 侧边栏
with st.sidebar:
    st.title("⚙️ 设置")
    report_type = st.radio("报告类型", ["daily", "weekly"], format_func=lambda x: "日报" if x=="daily" else "周报")
    st.markdown("---")
    st.info("💡 提示：用逗号或句号分隔不同任务")

# 主输入区
input_text = st.text_area("📝 工作记录", height=200, placeholder="例如：修复登录bug，开会讨论需求，写技术文档")

col1, col2 = st.columns(2)
with col1:
    generate_btn = st.button("🚀 生成报告", type="primary", use_container_width=True)
with col2:
    clear_btn = st.button("🗑️ 清空", use_container_width=True)

if generate_btn and input_text:
    with st.spinner("生成中..."):
        result = generator.generate(input_text, report_type)
        if result["success"]:
            st.success("✅ 生成成功")
            st.markdown('<div class="report-box">' + result["report"].replace("\\n", "<br>") + '</div>', unsafe_allow_html=True)
            st.download_button("📥 下载报告", result["report"], file_name=f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md")
        else:
            st.error(result["error"])
elif generate_btn:
    st.warning("请输入工作记录")

st.markdown("---")
st.markdown("<p style='text-align:center;color:gray'>📱 手机/电脑自适应 | 数据不上传</p>", unsafe_allow_html=True)
'''

# 写入文件
with open("app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_app_code)

print("✅ app.py 文件已创建成功！")
print(f"📁 位置: /Users/daisy/Documents/DaisyAI_Project/app.py")
print("")
print("下一步：在 Terminal 中运行以下命令启动应用：")
print("cd /Users/daisy/Documents/DaisyAI_Project")
print("conda activate DaisyEnv")
print("streamlit run app.py")

✅ app.py 文件已创建成功！
📁 位置: /Users/daisy/Documents/DaisyAI_Project/app.py

下一步：在 Terminal 中运行以下命令启动应用：
cd /Users/daisy/Documents/DaisyAI_Project
conda activate DaisyEnv
streamlit run app.py


In [5]:
# 在 Jupyter Notebook 中运行这个 Cell，创建所有必需文件

import os

# 1. 确保 requirements.txt 存在
requirements_content = """streamlit>=1.28.0
pandas>=2.0.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_content)
print("✅ requirements.txt 已创建")

# 2. 创建 .gitignore（排除不必要的文件）
gitignore_content = """__pycache__/
*.pyc
.DS_Store
.venv/
DaisyEnv/
*.pyo
*.pyd
.Python
env/
venv/
ENV/
.idea/
.vscode/
"""

with open(".gitignore", "w") as f:
    f.write(gitignore_content)
print("✅ .gitignore 已创建")

# 3. 创建 README.md
readme_content = '# 🤖 AI 智能日报生成器\n\n一个基于 Streamlit 的智能日报/周报生成工具，手机电脑都能用。\n\n## ✨ 在线使用\n\n点击链接即可使用（无需安装任何软件）：\n**[https://你的应用名.streamlit.app](https://你的应用名.streamlit.app)**\n\n## 🎯 功能特点\n\n- 📱 **响应式设计**：手机、平板、电脑完美适配\n- 🎯 **智能分类**：自动识别工作内容并分类\n- 📊 **支持日报/周报**：一键切换报告类型\n- 💾 **导出功能**：下载 Markdown 格式报告\n- 🔒 **隐私安全**：数据不上传，完全在浏览器处理\n\n## 📝 使用示例\n\n**输入：**\n```\n修复登录bug，开会讨论需求，写技术文档，明天准备发布\n```\n\n**输出：**\n- 今日完成：修复登录bug、开会讨论需求、写技术文档\n- 明日计划：准备发布\n\n## 🛠️ 本地运行\n\n```bash\ngit clone https://github.com/你的用户名/DailyReportGenerator.git\ncd DailyReportGenerator\npip install -r requirements.txt\nstreamlit run app.py\n```\n\n## 📄 License\n\nMIT\n'

with open("README.md", "w") as f:
    f.write(readme_content)
print("✅ README.md 已创建")

# 4. 检查 app.py 是否存在
if os.path.exists("app.py"):
    print("✅ app.py 已存在")
    size = os.path.getsize("app.py")
    print(f"   文件大小: {size} 字节")
else:
    print("❌ app.py 不存在，请先运行之前的创建代码")
    
print("\n" + "="*50)
print("📁 当前目录文件列表:")
for file in os.listdir("."):
    if file.endswith((".py", ".txt", ".md", ".gitignore")):
        print(f"   - {file}")

✅ requirements.txt 已创建
✅ .gitignore 已创建
✅ README.md 已创建
✅ app.py 已存在
   文件大小: 4399 字节

📁 当前目录文件列表:
   - requirements.txt
   - report_gen.py
   - README.md
   - .gitignore
   - app.py


In [6]:
# 检查文件
import os

files_to_check = ["app.py", "requirements.txt", ".gitignore", "README.md"]

print("📁 文件检查:")
print("="*40)

for file in files_to_check:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print(f"✅ {file:20s} ({size:6d} bytes)")
    else:
        print(f"❌ {file:20s} (不存在)")

print("="*40)

# 显示当前目录所有文件
print("\n📂 当前目录所有文件:")
for f in os.listdir("."):
    if not f.startswith(".") or f == ".gitignore":
        print(f"   {f}")

📁 文件检查:
✅ app.py               (  4399 bytes)
✅ requirements.txt     (    32 bytes)
✅ .gitignore           (    98 bytes)
✅ README.md            (  1057 bytes)

📂 当前目录所有文件:
   requirements.txt
   Demo_Report_Generator.ipynb
   LoveChatAI.html
   report_gen.py
   __pycache__
   README.md
   .gitignore
   app.py
   LoveChatAI.ipynb
